<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); border-radius: 15px; margin: 10px 0;'>
  <h1 style='color: #fff; margin: 0 0 8px 0; font-size: 2.2em;'>🎥 LTX-Video Generator</h1>
  <h3 style='color: #c0c0ff; margin: 0 0 5px 0; font-weight: 400;'>Lightning AI Studio | Wan2GP + mmgp</h3>
  <p style='color: #aaa; margin: 0;'>Text-to-Video & Image-to-Video | Q4_K_M GGUF</p>
</div>

---

### 🚀 Quick Start
1. **GPU Studio → T4** (30 GB RAM required)
2. **Cell 1** = install + download model (~17.8 GB, ~10 min)
3. **Cell 2** = launch Gradio UI

> ⚠️ If `import torch` fails, open a **fresh** T4 instance (Lightning AI instances are inconsistent)

In [ ]:
# @title ⚙️ Setup & Download Model (~17.8 GB)
import os, sys, subprocess
from pathlib import Path

HOME = os.path.expanduser('~')

print('[1/3] Installing dependencies...')
# Don't touch conda! Just pip install (skip torch - use system version)
!pip install -q huggingface_hub gradio

# Verify torch works (if not, open a FRESH T4 instance!)
import torch
print(f'  PyTorch {torch.__version__} OK')

print('[2/3] Installing Wan2GP + mmgp...')
%cd {HOME}
if not os.path.exists('Wan2GP'):
    !git clone -q https://github.com/deepbeepmeep/Wan2GP.git
%cd Wan2GP
!pip install -q -r requirements.txt
!pip install -q mmgp

print('[3/3] Downloading Q4_K_M model (~17.8 GB)...')
from huggingface_hub import hf_hub_download

def dl(repo, filename, dest_dir, dest_name=None):
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    fp = os.path.join(dest_dir, dest_name or filename)
    if not os.path.exists(fp):
        hf_hub_download(repo_id=repo, filename=filename, local_dir=dest_dir, local_dir_use_symlinks=False)
        tmp = os.path.join(dest_dir, filename)
        if tmp != fp and os.path.exists(tmp):
            os.rename(tmp, fp)
    else:
        print(f'  ✓ Already exists: {dest_name or filename}')

dl('unsloth/LTX-2.3-GGUF', 'ltx-2.3-22b-dev-Q4_K_M.gguf', f'{HOME}/models')
dl('Dampfinchen/google-gemma-3-12b-it-qat-q4_0-gguf-small-fix', 'gemma-3-12b-it-q4_0_s.gguf', f'{HOME}/models')
dl('unsloth/LTX-2.3-GGUF', 'text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors', f'{HOME}/models')
dl('unsloth/LTX-2.3-GGUF', 'vae/ltx-2.3-22b-dev_video_vae.safetensors', f'{HOME}/models')
dl('unsloth/LTX-2.3-GGUF', 'vae/ltx-2.3-22b-dev_audio_vae.safetensors', f'{HOME}/models')
dl('Lightricks/LTX-2.3', 'ltx-2.3-spatial-upscaler-x2-1.0.safetensors', f'{HOME}/models')
dl('Lightricks/LTX-2.3', 'ltx-2.3-22b-distilled-lora-384.safetensors', f'{HOME}/models')

print('✓ Ready! Run Cell 2 to launch.')

In [ ]:
# @title 🚀 Launch Gradio UI
import os, sys, subprocess, time, json, urllib.request, socket, threading
from pathlib import Path

HOME = os.path.expanduser('~')
MODEL_DIR = f'{HOME}/models'
WAN2GP = f'{HOME}/Wan2GP'

os.chdir(WAN2GP)

# Build mmgp offload config
config = {
    "offload_count": 4,
    "offload_fraction": 0.99,
    "offload_fallback": True,
    "offload_mode": "fraction",
    "verbose": True
}

config_path = f'{HOME}/mmgp_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f)

# Start Wan2GP server with mmgp - capture output for debugging
log_path = f'{HOME}/wan2gp_log.txt'
log_file = open(log_path, 'w')

cmd = [
    sys.executable, '-m', 'wan2gp.server',
    '--ckpt', f'{MODEL_DIR}/ltx-2.3-22b-dev-Q4_K_M.gguf',
    '--mmgp-config', config_path,
    '--text-encoder', f'{MODEL_DIR}/gemma-3-12b-it-q4_0_s.gguf',
    '--embeddings-connector', f'{MODEL_DIR}/ltx-2.3-22b-dev_embeddings_connectors.safetensors',
    '--video-vae', f'{MODEL_DIR}/ltx-2.3-22b-dev_video_vae.safetensors',
    '--spatial-upscaler', f'{MODEL_DIR}/ltx-2.3-spatial-upscaler-x2-1.0.safetensors',
    '--lora', f'{MODEL_DIR}/ltx-2.3-22b-distilled-lora-384.safetensors',
    '--port', '5000',
    '--host', '127.0.0.1'
]

proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

print('Booting Wan2GP server (checking logs)...')
for i in range(60):
    time.sleep(3)
    try:
        with socket.socket() as s:
            s.settimeout(2)
            if s.connect_ex(('127.0.0.1', 5000)) == 0:
                print('Server ready!')
                break
    except:
        pass
    if i % 5 == 0:
        # Show last few lines of log
        with open(log_path, 'r') as lf:
            lines = lf.readlines()
            for line in lines[-3:]:
                print(f'  [log] {line.rstrip()}')
else:
    print('Server failed to start. Full log:')
    with open(log_path, 'r') as lf:
        print(lf.read())
    raise RuntimeError('Wan2GP server did not start')

# ---------- Gradio UI ----------
import gradio as gr

def t2v(prompt, width, height, length_sec, seed, steps, cfg):
    safe_w = max(256, round(width / 32) * 32)
    safe_h = max(256, round(height / 32) * 32)
    frames = int(length_sec * 24)
    
    payload = {
        "prompt": prompt,
        "width": safe_w,
        "height": safe_h,
        "num_frames": frames,
        "seed": int(seed),
        "steps": int(steps),
        "cfg_scale": float(cfg),
        "mode": "t2v"
    }
    
    req = urllib.request.Request(
        'http://127.0.0.1:5000/generate',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'}
    )
    res = json.loads(urllib.request.urlopen(req).read())
    
    if 'video' in res:
        return res['video']
    return None

def i2v(image, prompt, width, height, length_sec, seed, steps, cfg):
    safe_w = max(256, round(width / 32) * 32)
    safe_h = max(256, round(height / 32) * 32)
    frames = int(length_sec * 24)
    
    img_path = f'{HOME}/input_image.png'
    image.save(img_path)
    
    payload = {
        "prompt": prompt,
        "image": img_path,
        "width": safe_w,
        "height": safe_h,
        "num_frames": frames,
        "seed": int(seed),
        "steps": int(steps),
        "cfg_scale": float(cfg),
        "mode": "i2v"
    }
    
    req = urllib.request.Request(
        'http://127.0.0.1:5000/generate',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'}
    )
    res = json.loads(urllib.request.urlopen(req).read())
    
    if 'video' in res:
        return res['video']
    return None

with gr.Blocks(title='LTX-Video Generator') as demo:
    gr.Markdown('# 🎥 LTX-Video Generator\nLightning AI T4 | Wan2GP + mmgp | Q4_K_M')
    
    with gr.Tabs():
        with gr.TabItem('📝 Text-to-Video'):
            with gr.Row():
                with gr.Column(scale=1):
                    prompt = gr.Textbox(label='Prompt', value='A cinematic shot of a futuristic sports car driving through a neon-lit cyberpunk city at night.', lines=3)
                    with gr.Row():
                        width = gr.Slider(256, 1280, 832, step=32, label='Width')
                        height = gr.Slider(256, 720, 480, step=32, label='Height')
                    length_sec = gr.Slider(1, 10, 4, step=1, label='Duration (seconds)')
                    with gr.Accordion('Advanced', open=False):
                        seed = gr.Number(42, label='Seed', precision=0)
                        steps = gr.Slider(10, 50, 25, step=1, label='Steps')
                        cfg = gr.Slider(1.0, 10.0, 3.0, step=0.5, label='CFG Scale')
                    btn = gr.Button('Generate', variant='primary')
                with gr.Column(scale=2):
                    video_out = gr.Video(label='Output', height=400)
            btn.click(t2v, [prompt, width, height, length_sec, seed, steps, cfg], [video_out])
        
        with gr.TabItem('🖼️ Image-to-Video'):
            with gr.Row():
                with gr.Column(scale=1):
                    img_in = gr.Image(label='Input Image', type='pil')
                    prompt_i2v = gr.Textbox(label='Prompt', value='The subject smiles and waves at the camera.', lines=2)
                    with gr.Row():
                        w2 = gr.Slider(256, 1280, 832, step=32, label='Width')
                        h2 = gr.Slider(256, 720, 480, step=32, label='Height')
                    len2 = gr.Slider(1, 10, 3, step=1, label='Duration (seconds)')
                    with gr.Accordion('Advanced', open=False):
                        seed2 = gr.Number(42, label='Seed', precision=0)
                        steps2 = gr.Slider(10, 50, 25, step=1, label='Steps')
                        cfg2 = gr.Slider(1.0, 10.0, 3.0, step=0.5, label='CFG Scale')
                    btn2 = gr.Button('Generate', variant='primary')
                with gr.Column(scale=2):
                    video_out2 = gr.Video(label='Output', height=400)
            btn2.click(i2v, [img_in, prompt_i2v, w2, h2, len2, seed2, steps2, cfg2], [video_out2])

demo.launch(share=True)